# Data-derived wavelength constraints in PGMUVI

This maintained tutorial explains how PGMUVI derives, transforms, applies, and diagnoses wavelength-dependent parameter constraints for sparse multi-band light curves. The examples are designed around long-period variables (LPVs), use **linear flux**, and remain advisory: constraint behavior informs plausibility and numerical reliability but does not perform automatic model selection.

A constraint is a numerical and scientific domain restriction. It does not mean that a parameter is known precisely, and a fitted value near a bound is not by itself evidence for a physical interpretation.


## Execution policy

Restart the kernel and use **Run All** to reproduce the default path. It is deterministic, requires no internet connection, writes no files, and does not train a GP.

`RUN_REDUCED_SYNTHETIC_FITS` is deliberately `False`. Turning it on runs a small illustrative fit, not the full 420-run calibration. The LPV demonstration uses `fit_strategy="consensus"`, `time_kernel_type="quasi_periodic"`, and `learn_additional_noise=True` because this is the finite-coherence, recurring-timescale path exercised by the D1/D2 validation. It is a tested default hypothesis, not a claim that every LPV is quasi-periodic. Strictly periodic, aperiodic, or multi-periodic time models can be more appropriate when supported by the data.

The wavelength-constraint machinery is conceptually separate from the time-kernel choice: it restricts wavelength-dependent mean and covariance parameters in the coordinate system where those parameters are applied.


In [ ]:
from pprint import pprint

import numpy as np
import torch

from pgmuvi.spectral_mixture_ard import (
    ARD_COORDINATE_ORDER,
    build_dimension_aware_sm_ard_estimates,
)
from pgmuvi.spectral_mixture_ard_diagnostics import diagnose_spectral_mixture_ard
from pgmuvi.wavelength_estimation import (
    build_wavelength_estimation_context,
    build_wavelength_mean_estimation_context,
)
from pgmuvi.wavelength_hypotheses import (
    LPV_WAVELENGTH_MODEL_PRIORITY,
    describe_wavelength_model_hypothesis,
)
from pgmuvi.wavelength_validation_robustness import (
    canonical_synthetic_wavelength_robustness_cases,
)

SEED = 0
RUN_REDUCED_SYNTHETIC_FITS = False
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_default_dtype(torch.float64)


## 1. Scientific motivation and model hypotheses

Sparse LPV light curves may have unequal numbers of observations by band, missing edge or interior bands, large wavelength gaps, heteroscedastic uncertainties, and incomplete phase coverage. Data-derived constraints prevent the optimizer from treating every mathematically allowed wavelength scale as equally plausible.

The complete model configurations separate several questions:

- **time dependence**: periodic, quasi-periodic, aperiodic, or multi-component behavior;
- **wavelength-dependent covariance**: whether variability remains correlated across wavelength;
- **wavelength-dependent mean**: whether the baseline flux changes with wavelength;
- **separable covariance**: a product of time and wavelength kernels;
- **joint non-separable covariance**: a single two-dimensional covariance over time and wavelength;
- **evidence strength**: strong, weak, or insufficient wavelength information.

The package taxonomy describes these hypotheses without ranking or selecting them.


In [ ]:
taxonomy_rows = []
for model_name in LPV_WAVELENGTH_MODEL_PRIORITY:
    item = describe_wavelength_model_hypothesis(model_name).to_dict()
    taxonomy_rows.append({
        "model": item["model"],
        "priority": item["lpv_advisory_priority"],
        "role": item["role"],
        "mean_structure": item["mean_structure"],
        "covariance_structure": item["covariance_structure"],
        "mean_wavelength_dependent": item["mean_wavelength_dependent"],
        "covariance_wavelength_dependent": item[
            "covariance_wavelength_dependent"
        ],
        "covariance_separable": item["covariance_separable"],
        "summary": item["summary"],
    })

taxonomy_rows


The maintained LPV priority order is:

1. `2DWavelengthDependent`: quadratic wavelength mean plus separable smooth wavelength covariance;
2. `2DDustMean`: dust-like physical mean plus wavelength-dependent covariance;
3. `2DPowerLawMean`: power-law physical mean plus wavelength-dependent covariance;
4. `2DSeparable`: constant mean plus separable time × wavelength covariance;
5. `2D`: joint non-separable spectral-mixture baseline.

These labels describe complete mean-plus-covariance configurations. A score difference cannot automatically be attributed to one isolated physical mechanism.


## 2. Deterministic reference data and wavelength coordinates

The D2 registry supplies deterministic, truth-preserving synthetic cases. The reference below uses six physical wavelengths in micrometres, a recurring 500-day timescale with finite coherence, and a separable wavelength covariance. The light curve remains in linear flux.


In [ ]:
robustness_cases = {
    case.scenario.scenario_id: case
    for case in canonical_synthetic_wavelength_robustness_cases()
}
reference_case = robustness_cases["d2-reference-separable-moderate"]
reference_lc = reference_case.to_lightcurve()

raw_inputs = np.column_stack([
    reference_case.time_values,
    reference_case.wavelength_values,
])
raw_inputs_tensor = torch.as_tensor(raw_inputs, dtype=torch.float64)
model_inputs_tensor = reference_lc.transform_x(raw_inputs_tensor)
round_trip_tensor = reference_lc.xtransform.inverse(model_inputs_tensor)

round_trip_max_absolute_error = float(
    torch.max(torch.abs(round_trip_tensor - raw_inputs_tensor))
)
round_trip_scale = float(torch.max(torch.abs(raw_inputs_tensor)))
round_trip_max_relative_error = (
    round_trip_max_absolute_error / max(round_trip_scale, 1.0)
)
assert round_trip_max_relative_error < 1.0e-12

coordinate_provenance = reference_case.scenario.truth.coordinate_transforms
coordinate_summary = {
    "physical_wavelength_unit": "micrometre",
    "raw_coordinate_space": "time_days_and_physical_wavelength",
    "model_coordinate_space": reference_lc.xtransform.__class__.__name__,
    "transform_provenance": coordinate_provenance,
    "raw_unique_wavelengths": sorted(set(reference_case.wavelength_values)),
    "model_unique_wavelengths": sorted(
        set(model_inputs_tensor[:, 1].detach().cpu().numpy().tolist())
    ),
    "round_trip_max_absolute_error": round_trip_max_absolute_error,
    "round_trip_max_relative_error": round_trip_max_relative_error,
}
coordinate_summary


A kernel lengthscale must be constrained in the same coordinate used by that kernel. Under an affine MinMax transform, physical wavelength is shifted and rescaled into the model coordinate. Lengthscales and interval widths use only the scale factor; coordinate origins must not be applied to scale parameters.

Physical dust and power-law means deliberately reconstruct positive physical wavelength. The flexible quadratic mean instead uses the model-input wavelength coordinate.


## 3. Data-derived wavelength evidence

`build_wavelength_estimation_context` separates wavelength sampling from robust per-band flux summaries. The returned records are diagnostics; they do not modify a model.


In [ ]:
wavelength_diagnostics, band_diagnostics = (
    build_wavelength_estimation_context(
        reference_case.wavelength_values,
        reference_case.observed_flux,
        reference_case.uncertainties,
        reference_case.band_labels,
        min_points_per_band=3,
    )
)
wavelength_summary = wavelength_diagnostics.to_dict()

sampling_evidence = {
    key: wavelength_summary[key]
    for key in [
        "available",
        "n_observations",
        "n_distinct_wavelengths",
        "n_usable_bands",
        "wavelength_min",
        "wavelength_max",
        "wavelength_span",
        "adjacent_spacings",
        "minimum_adjacent_spacing",
        "median_adjacent_spacing",
        "largest_gap",
        "largest_gap_ratio_to_median_spacing",
        "spacing_ratio_max_to_min",
        "coverage_class",
        "excluded_bands",
    ]
}
sampling_evidence


In [ ]:
per_band_evidence = []
for band_name, item in band_diagnostics.items():
    per_band_evidence.append({
        "band": band_name,
        "wavelength": item.wavelength,
        "n_points": item.n_points,
        "median_uncertainty": item.median_uncertainty,
        "median_flux": item.median_flux,
        "robust_scatter": item.robust_scatter,
        "noise_corrected_robust_scatter": item.noise_corrected_robust_scatter,
        "raw_half_amplitude_q02_5_q97_5": (
            item.raw_half_amplitude_q02_5_q97_5
        ),
        "raw_half_amplitude_q05_q95": item.raw_half_amplitude_q05_q95,
        "raw_half_amplitude_q10_q90": item.raw_half_amplitude_q10_q90,
        "fractional_half_amplitude_q05_q95": (
            item.fractional_half_amplitude_q05_q95
        ),
        "noise_corrected_half_amplitude_q05_q95": (
            item.noise_corrected_half_amplitude_q05_q95
        ),
        "usable": item.metadata.get("usable_for_wavelength_estimation"),
    })

per_band_evidence


The evidence should be labeled by role rather than collapsed into one verdict:

- **observed facts**: wavelengths, band labels, sample counts, reported uncertainties;
- **derived statistics**: spacing, robust medians, quantile amplitudes, scatter, and gap ratios;
- **heuristic interpretation**: apparent smoothness, monotonicity, turning-point structure, or weak coverage;
- **formal comparison results**: recovery metrics or calibrated D2 classes, when available;
- **workflow warnings**: excluded bands, near-bound parameters, or consensus rejection;
- **future-work limitations**: unsupported physical kernels, multiple periods, or wavelength-dependent lags.


In [ ]:
evidence_role_examples = [
    {
        "role": "observed_fact",
        "example": "six unique physical wavelengths with per-band counts",
    },
    {
        "role": "derived_statistic",
        "example": "largest-gap ratio and robust Q05-Q95 half-amplitude",
    },
    {
        "role": "heuristic_interpretation",
        "example": wavelength_summary["amplitude_monotonicity_class"],
    },
    {
        "role": "formal_comparison_result",
        "example": "PR133 paired multi-seed recovery classification",
    },
    {
        "role": "workflow_warning",
        "example": "near-bound wavelength scale is not achromatic evidence",
    },
    {
        "role": "future_work_limitation",
        "example": "no validated wavelength-dependent lag kernel",
    },
]
evidence_role_examples


## 4. Wavelength-kernel constraint derivation

The separable families derive a raw-coordinate wavelength lengthscale from usable wavelength sampling. The current recommendation combines the median adjacent spacing with total span, then forms a finite interval using the minimum spacing, largest gap, and span. Sparse-band adaptations are encoded in the recommendation method and provenance.


In [ ]:
raw_lengthscale_record = {
    "initial": wavelength_summary["recommended_lengthscale_initial"],
    "bounds": wavelength_summary["recommended_lengthscale_bounds"],
    "method": wavelength_summary["recommendation_method"],
    "coordinate_space": wavelength_summary["coordinate_space"],
    "units": wavelength_summary["metadata"]["lengthscale_units"],
    "minimum_spacing": wavelength_summary["minimum_adjacent_spacing"],
    "median_spacing": wavelength_summary["median_adjacent_spacing"],
    "largest_gap": wavelength_summary["largest_gap"],
    "span": wavelength_summary["wavelength_span"],
}

raw_initial = torch.tensor(
    [[0.0, raw_lengthscale_record["initial"]]],
    dtype=torch.float64,
)
raw_bounds = torch.tensor(
    [
        [0.0, raw_lengthscale_record["bounds"][0]],
        [0.0, raw_lengthscale_record["bounds"][1]],
    ],
    dtype=torch.float64,
)
model_initial = float(
    reference_lc.xtransform.transform(raw_initial, shift=False)[0, 1]
)
model_bounds = tuple(
    reference_lc.xtransform.transform(raw_bounds, shift=False)[:, 1]
    .detach()
    .cpu()
    .numpy()
    .tolist()
)

lengthscale_coordinate_record = {
    "raw_physical_scale": raw_lengthscale_record,
    "model_coordinate_initial": model_initial,
    "model_coordinate_bounds": model_bounds,
    "transform": reference_lc.xtransform.__class__.__name__,
    "scale_only_transform": True,
    "constraint_source": "wavelength_estimation_context",
    "initialization_source": "data_derived_wavelength_sampling",
}
lengthscale_coordinate_record


Overly broad bounds allow weakly identified lengthscales to develop long-tailed recovery errors. This matters when wavelength coverage has large holes, when one edge band is missing, or when sample counts are highly uneven. The upper bound is a numerical domain limit, not an achromaticity threshold.


In [ ]:
scenario_ids = [
    "d2-reference-separable-moderate",
    "d2-reference-wavelength-quadratic-strong-turning",
    "d2-reference-joint-sm-ard-moderate",
    "d2-joint-sm-sparse-independent",
    "d2-missing-blue-edge-band",
    "d2-missing-red-edge-band",
    "d2-missing-interior-band",
    "d2-large-wavelength-gap",
    "d2-uneven-band-counts",
    "d2-longer-sparse-baseline",
    "d2-insufficient-per-band-sampling",
]

scenario_constraint_rows = []
for scenario_id in scenario_ids:
    case = robustness_cases[scenario_id]
    diagnostics, bands = build_wavelength_estimation_context(
        case.wavelength_values,
        case.observed_flux,
        case.uncertainties,
        case.band_labels,
        min_points_per_band=3,
    )
    record = diagnostics.to_dict()
    counts = sorted(
        int(item.n_points or 0)
        for item in bands.values()
    )
    scenario_constraint_rows.append({
        "scenario": scenario_id,
        "usable_bands": record["n_usable_bands"],
        "per_band_count_min": min(counts) if counts else None,
        "per_band_count_max": max(counts) if counts else None,
        "span": record["wavelength_span"],
        "largest_gap": record["largest_gap"],
        "largest_gap_ratio": record[
            "largest_gap_ratio_to_median_spacing"
        ],
        "initial": record["recommended_lengthscale_initial"],
        "lower_bound": (
            record["recommended_lengthscale_bounds"][0]
            if record["recommended_lengthscale_bounds"]
            else None
        ),
        "upper_bound": (
            record["recommended_lengthscale_bounds"][1]
            if record["recommended_lengthscale_bounds"]
            else None
        ),
        "coverage_class": record["coverage_class"],
        "excluded_bands": record["excluded_bands"],
    })

scenario_constraint_rows


## 5. Wavelength-dependent mean constraints

The covariance lengthscale and wavelength-dependent mean parameters are different objects. Robust per-band medians supply model-ready recommendations for `2DWavelengthDependent`, `2DDustMean`, and `2DPowerLawMean`.

The quadratic mean uses model wavelength. Dust and power-law means use positive physical wavelength while retaining the transformed training-target coordinate. Numerically acceptable parameters are not automatically unique physical measurements.


In [ ]:
model_inputs = model_inputs_tensor.detach().cpu().numpy()
model_fluxes = reference_lc.transform_y(reference_lc.ydata).detach().cpu().numpy()
mean_diagnostics = build_wavelength_mean_estimation_context(
    reference_case.wavelength_values,
    model_inputs[:, 1],
    model_fluxes,
    reference_case.band_labels,
    min_points_per_band=3,
)

mean_constraint_rows = []
for model_name in [
    "2DWavelengthDependent",
    "2DDustMean",
    "2DPowerLawMean",
]:
    recommendation = mean_diagnostics.recommendations[model_name]
    for parameter_name, initial_value in recommendation.get(
        "initial_values", {}
    ).items():
        mean_constraint_rows.append({
            "model": model_name,
            "parameter": parameter_name,
            "coordinate_basis": recommendation["coordinate_basis"],
            "initial_value": initial_value,
            "constraint": recommendation.get("constraints", {}).get(
                parameter_name
            ),
            "fit_rmse": recommendation.get("fit_rmse"),
            "available": recommendation.get("available"),
            "reason": recommendation.get("reason"),
            "prior": "inspect fitted model; no data-derived prior is implied",
        })

mean_constraint_rows


A monotonic profile and a turning-point profile can have similar wavelength coverage while supporting different mean-shape interpretations. The next comparison derives both from the maintained scenario registry. The quadratic recommendation is a flexible numerical description; it does not establish a unique physical mechanism.


In [ ]:
mean_structure_case_ids = [
    "d2-reference-wavelength-quadratic-moderate",
    "d2-reference-wavelength-quadratic-strong-turning",
]

mean_structure_rows = []
for scenario_id in mean_structure_case_ids:
    case = robustness_cases[scenario_id]
    lightcurve = case.to_lightcurve()
    raw_case_inputs = np.column_stack([
        case.time_values,
        case.wavelength_values,
    ])
    model_case_inputs = lightcurve.transform_x(
        torch.as_tensor(raw_case_inputs, dtype=torch.float64)
    ).detach().cpu().numpy()
    model_case_fluxes = lightcurve.transform_y(
        lightcurve.ydata
    ).detach().cpu().numpy()
    case_wavelength_diagnostics, _ = build_wavelength_estimation_context(
        case.wavelength_values,
        case.observed_flux,
        case.uncertainties,
        case.band_labels,
        min_points_per_band=3,
    )
    case_mean_diagnostics = build_wavelength_mean_estimation_context(
        case.wavelength_values,
        model_case_inputs[:, 1],
        model_case_fluxes,
        case.band_labels,
        min_points_per_band=3,
    )
    quadratic = case_mean_diagnostics.recommendations[
        "2DWavelengthDependent"
    ]
    mean_structure_rows.append({
        "scenario": scenario_id,
        "median_flux_monotonicity": (
            case_wavelength_diagnostics.median_flux_monotonicity_class
        ),
        "amplitude_monotonicity": (
            case_wavelength_diagnostics.amplitude_monotonicity_class
        ),
        "quadratic_fit_degree": quadratic.get("fit_degree"),
        "quadratic_initial_weights": quadratic.get(
            "initial_values", {}
        ).get("mean_module.weights"),
        "quadratic_fit_rmse": quadratic.get("fit_rmse"),
        "available": quadratic.get("available"),
    })

mean_structure_rows


## 6. Constraint registration and initialization integrity

The safe workflow is:

1. derive the final proposed constraint;
2. intersect it with package defaults or an existing user interval where appropriate;
3. register the final constraint;
4. assign the initial constrained value;
5. preserve a value that is already valid;
6. clamp or reject an invalid value deliberately rather than silently;
7. record the constraint and initialization provenance;
8. prevent later initialization stages from overwriting the accepted value.

Assigning a value before registering its final interval can transform it through the wrong raw-parameter map and previously caused invalid initializations, including interval errors for wavelength-mean parameters.


## 7. Independent time and wavelength ARD constraints for joint `2D`

The joint non-separable `2D` spectral-mixture model uses a fixed coordinate order:

- ARD index 0 = `temporal_frequency`;
- ARD index 1 = `wavelength_frequency`.

Time-derived limits must never be broadcast onto index 1. Period or source-type constraints alter only the temporal dimension.


In [ ]:
assert tuple(ARD_COORDINATE_ORDER) == (
    "temporal_frequency",
    "wavelength_frequency",
)

ard_estimates = build_dimension_aware_sm_ard_estimates(
    raw_inputs=raw_inputs,
    model_inputs=model_inputs,
    num_mixtures=2,
    wavelength_diagnostics=wavelength_diagnostics,
)

ard_estimate_rows = []
for parameter_name in ["mixture_means", "mixture_scales"]:
    raw_record = ard_estimates["raw_coordinate"][parameter_name]
    model_record = ard_estimates["model_coordinate"][parameter_name]
    for component_index in range(ard_estimates["num_mixtures"]):
        for dimension_index, dimension_label in enumerate(
            ARD_COORDINATE_ORDER
        ):
            ard_estimate_rows.append({
                "parameter": parameter_name,
                "component_index": component_index,
                "ard_dimension_index": dimension_index,
                "ard_dimension_label": dimension_label,
                "raw_coordinate_initial": raw_record["initial_value"][
                    component_index
                ][0][dimension_index],
                "raw_coordinate_lower": raw_record["constraint_lower"][0][0][
                    dimension_index
                ],
                "raw_coordinate_upper": raw_record["constraint_upper"][0][0][
                    dimension_index
                ],
                "model_coordinate_initial": model_record["initial_value"][
                    component_index
                ][0][dimension_index],
                "model_coordinate_lower": model_record[
                    "constraint_lower"
                ][0][0][dimension_index],
                "model_coordinate_upper": model_record[
                    "constraint_upper"
                ][0][0][dimension_index],
            })

ard_estimate_rows


The temporal and wavelength rows have different initial values and different interval widths. This is the required dimension-aware behavior. The full parameter report also retains GPyTorch raw values so users can distinguish unconstrained raw parameters, transformed model-coordinate values, and reconstructed raw-input-coordinate values.


## 8. Reusable fitted-constraint diagnostic table

The functions below flatten maintained public reports into one advisory schema. They do not classify scientific compatibility and do not select a model.


In [ ]:
DIAGNOSTIC_COLUMNS = [
    "model",
    "parameter_name",
    "component_index",
    "ard_dimension_index",
    "ard_dimension_label",
    "physical_value",
    "transformed_value",
    "gpytorch_raw_parameter_value",
    "lower_bound",
    "upper_bound",
    "normalized_distance_to_lower_bound",
    "normalized_distance_to_upper_bound",
    "nearest_bound_distance",
    "near_bound",
    "at_bound",
    "constraint_source",
    "initialization_source",
    "data_derived_provenance",
]


def json_ready_parameter(value):
    if value is None:
        return None
    if isinstance(value, torch.Tensor):
        array = value.detach().cpu().numpy()
        return float(array) if array.ndim == 0 else array.tolist()
    return value


def flatten_parameter_workflow(
    model_name,
    report,
    model_coordinate_parameters,
    physical_coordinate_parameters,
    raw_parameters,
):
    rows = []
    entries = list((report or {}).get("applied", []))
    entries.extend((report or {}).get("skipped", []))
    for item in entries:
        parameter_name = item.get("parameter")
        wavelength_provenance = item.get("wavelength_estimate_provenance")
        mean_provenance = item.get("wavelength_mean_estimate_provenance")
        provenance = wavelength_provenance or mean_provenance or {}
        effective = provenance.get("effective_constraint") or {}
        rows.append({
            "model": model_name,
            "parameter_name": parameter_name,
            "component_index": None,
            "ard_dimension_index": None,
            "ard_dimension_label": None,
            "physical_value": json_ready_parameter(
                physical_coordinate_parameters.get(parameter_name)
            ),
            "transformed_value": json_ready_parameter(
                model_coordinate_parameters.get(parameter_name)
            ),
            "gpytorch_raw_parameter_value": json_ready_parameter(
                raw_parameters.get(parameter_name)
            ),
            "lower_bound": effective.get("lower_bound"),
            "upper_bound": effective.get("upper_bound"),
            "normalized_distance_to_lower_bound": None,
            "normalized_distance_to_upper_bound": None,
            "nearest_bound_distance": None,
            "near_bound": None,
            "at_bound": None,
            "constraint_source": item.get("constraint_action"),
            "initialization_source": (
                "parameter_workflow_value"
                if item.get("value_applied")
                else item.get("value_reason")
            ),
            "data_derived_provenance": provenance,
        })
    return rows


def flatten_sm_ard_diagnostics(model_name, diagnostics):
    rows = []
    parameters = (diagnostics or {}).get("parameters", {})
    for parameter_name in ["mixture_means", "mixture_scales"]:
        records = parameters.get(parameter_name, {}).get(
            "component_diagnostics", []
        )
        for item in records:
            near_bound = bool(item.get("near_lower") or item.get("near_upper"))
            at_bound = bool(item.get("at_lower") or item.get("at_upper"))
            rows.append({
                "model": model_name,
                "parameter_name": parameter_name,
                "component_index": item.get("component_index"),
                "ard_dimension_index": item.get("dimension_index"),
                "ard_dimension_label": item.get("dimension_name"),
                "physical_value": item.get("raw_input_coordinate_value"),
                "transformed_value": item.get("model_coordinate_value"),
                "gpytorch_raw_parameter_value": item.get(
                    "gpytorch_raw_parameter_value"
                ),
                "lower_bound": item.get("model_coordinate_lower_bound"),
                "upper_bound": item.get("model_coordinate_upper_bound"),
                "normalized_distance_to_lower_bound": item.get(
                    "normalized_distance_to_lower"
                ),
                "normalized_distance_to_upper_bound": item.get(
                    "normalized_distance_to_upper"
                ),
                "nearest_bound_distance": item.get(
                    "distance_to_nearest_bound"
                ),
                "near_bound": near_bound,
                "at_bound": at_bound,
                "constraint_source": "registered_spectral_mixture_interval",
                "initialization_source": "spectral_mixture_ard_provenance",
                "data_derived_provenance": {
                    "coordinate_transform_source": (
                        parameters.get(parameter_name, {}).get(
                            "coordinate_transform_source"
                        )
                    ),
                    "constraint_registered": (
                        parameters.get(parameter_name, {}).get(
                            "constraint_registered"
                        )
                    ),
                },
            })
    return rows


## 9. Optional reduced synthetic fits

The default run stops before optimization. Enabling the guard runs the models listed in `REDUCED_FIT_MODELS`; the default opt-in pair covers one wavelength-dependent mean model and the joint `2D` ARD baseline. The maintained case map also exposes `2DDustMean`, `2DPowerLawMean`, and `2DSeparable` so users can substitute those families without changing the diagnostic workflow. This reduced path does not reproduce the population calibration. Structured failures are retained as outcomes rather than converted into successful fits.


In [ ]:
reduced_fit_configuration = {
    "fit_strategy": "consensus",
    "time_kernel_type": "quasi_periodic",
    "learn_additional_noise": True,
    "training_iter": 40,
    "miniter": 10,
    "verbose": False,
}

reduced_fit_cases = {
    "2DWavelengthDependent": robustness_cases[
        "d2-reference-wavelength-quadratic-moderate"
    ],
    "2DDustMean": robustness_cases[
        "d2-reference-dust-moderate"
    ],
    "2DPowerLawMean": robustness_cases[
        "d2-reference-power-law-moderate"
    ],
    "2DSeparable": robustness_cases[
        "d2-reference-separable-moderate"
    ],
    "2D": robustness_cases[
        "d2-reference-joint-sm-ard-moderate"
    ],
}
REDUCED_FIT_MODELS = (
    "2DWavelengthDependent",
    "2D",
)

fit_results = {}
if RUN_REDUCED_SYNTHETIC_FITS:
    for model_name in REDUCED_FIT_MODELS:
        case = reduced_fit_cases[model_name]
        lightcurve = case.to_lightcurve()
        try:
            result = lightcurve.fit(
                model=model_name,
                **reduced_fit_configuration,
            )
            workflow_report = lightcurve.get_parameter_workflow_report()
            model_coordinate_parameters = lightcurve.get_parameters(
                raw=False,
                transform=False,
            )
            physical_coordinate_parameters = lightcurve.get_parameters(
                raw=False,
                transform=True,
            )
            raw_parameters = lightcurve.get_parameters(raw=True)
            rows = flatten_parameter_workflow(
                model_name,
                workflow_report,
                model_coordinate_parameters,
                physical_coordinate_parameters,
                raw_parameters,
            )
            ard_diagnostics = None
            if model_name == "2D":
                ard_diagnostics = diagnose_spectral_mixture_ard(
                    lightcurve,
                    boundary_tolerance_fraction=0.05,
                    at_bound_tolerance_fraction=1.0e-6,
                    absolute_tolerance=1.0e-12,
                )
                rows.extend(
                    flatten_sm_ard_diagnostics(
                        model_name,
                        ard_diagnostics,
                    )
                )
            fit_results[model_name] = {
                "status": "completed",
                "fit_result": result,
                "parameter_workflow": workflow_report,
                "ard_diagnostics": ard_diagnostics,
                "diagnostic_rows": rows,
            }
        except Exception as exception:
            fit_results[model_name] = {
                "status": "failed",
                "exception_type": type(exception).__name__,
                "exception_message": str(exception),
                "structured_failure_diagnostics": getattr(
                    exception,
                    "failure_diagnostics",
                    None,
                ),
            }
else:
    fit_results = {
        "status": "disabled",
        "reason": "Set RUN_REDUCED_SYNTHETIC_FITS=True explicitly.",
    }

fit_results


## 10. Near-bound, at-bound, and scientific interpretation

A **near-bound** value lies within a configured fractional distance of a bound. An **at-bound** value is equal to a bound within a stricter numerical tolerance. Upper-bound pressure, lower-bound pressure, and exact saturation should be reported separately.

A wavelength scale near its upper limit can indicate weak identifiability, sparse coverage, a broad interval, or an optimizer preference within the allowed domain. It does **not** prove achromaticity. Conversely, poor recovery can occur without a recorded boundary hit.


## 11. Compact PR133 calibration summary

The maintained empirical summary below records the canonical 20-seed D2 run without depending on `validation_outputs/` or a deleted local calibration directory.


In [ ]:
pr133_calibration_summary = {
    "source": "PR133 canonical D2 calibration, base seeds 0 through 19",
    "canonical_runs": 420,
    "completed_fits": 393,
    "recorded_failures": 27,
    "expected_failures": 20,
    "expected_failures_matched": 20,
    "structured_unexpected_consensus_rejections": 7,
    "reference_populations": 7,
    "robust_perturbations": 10,
    "degraded_perturbations": 1,
    "failure_boundaries": 2,
    "expected_failure_boundaries": 1,
    "degraded_case": "d2-large-wavelength-gap",
    "failure_boundary_cases": [
        "d2-uneven-band-counts",
        "d2-longer-sparse-baseline",
    ],
    "expected_failure_boundary_case": (
        "d2-insufficient-per-band-sampling"
    ),
}
pr133_calibration_summary


The three central empirical findings are:

1. **Uneven band counts can destroy wavelength-scale recovery.** Only 8 of 19 completed runs passed the configured wavelength-lengthscale criterion.
2. **A longer but sparse baseline can destroy period recovery.** Only 4 of 20 runs passed the period criterion; a longer calendar baseline does not compensate for insufficient information content.
3. **A large wavelength gap is strongly degraded.** Its lengthscale recovery passed 9 of 20 runs and showed long-tailed factor errors, even though it remained below the formal failure-boundary threshold.

The seven unexpected failures were structured consensus-stage rejections, not optimizer crashes. Six occurred at base seed 7 and one at seed 10. The seed-7 weak-dependence case exposed an upstream Lomb–Scargle/ACF disagreement: Lomb–Scargle recovered approximately the injected 500-day period while the ACF selected a spurious approximately 23-day timescale. That limitation belongs to consensus validation and must be discussed separately from wavelength-kernel optimization.


The joint `2D` reference and sparse perturbation both had temporal and wavelength ARD near-bound fractions of 1.0 while recovery remained robust and no parameters were recorded as exactly at the boundary. This is direct evidence that boundary pressure is not equivalent to recovery failure and must not be converted into an achromatic conclusion.


## 12. Advisory conclusion and next diagnostics

This notebook does **not** perform automatic model selection. Constraint diagnostics inform whether a fit is numerically credible and whether a model family remains scientifically plausible. They must be combined with fit quality, residual wavelength structure, period diagnostics, warnings, repeated-seed behavior, and source-specific scientific context.

Keep these outcomes distinct:

- **technical failure**: execution or optimization could not complete;
- **diagnostic unavailability**: the requested statistic could not be estimated from the available evidence;
- **scientific incompatibility**: a completed model is inconsistent with the observed structure or scientific assumptions.

Continue with the [wavelength-estimation API](../pgmuvi.wavelength_estimation.rst), [spectral-mixture ARD API](../pgmuvi.spectral_mixture_ard.rst), [ARD diagnostics API](../pgmuvi.spectral_mixture_ard_diagnostics.rst), [wavelength-model guide](../howto/wavelength_models.rst), [priors and constraints guide](../howto/priors_constraints.rst), [consensus-fitting guide](../howto/consensus_fitting.rst), [result-interpretation guide](../howto/interpreting_results.rst), and [robustness-calibration documentation](../pgmuvi.wavelength_validation_robustness_calibration.rst). The calibration page supplies the full D2 policy and provenance.
